# Mzansi Solar Forecasting: Planning Grid Purchases 30 Days Ahead

You're an energy analyst at the **Mzansi Textiles facility** — a 2.3 MW rooftop PV installation powering a textile manufacturing plant in Durban, KZN.
You have 6+ months of daily solar production and consumption data logged from the site's energy management system.

**The goal:** forecast the next 30 days of solar production to plan grid purchases and reduce demand charges.

**Kit data** (fetched over GitHub raw URLs — no clone needed):

| Source | Contents |
|---|---|
| `data/energy_balance.csv` | ~170 daily rows of solar production, consumption & grid purchases (Oct 2025 – May 2026) |
| `data/mzansi-consumption-monthly-agg.csv` | Multi-year monthly consumption aggregates (2023 – 2026) for seasonal context |

Pipeline: **load → explore → normalize to ODS-E → fit SARIMAX → 30-day forecast → grid purchase planning**.


In [ ]:
!pip install -q odse pandas matplotlib statsmodels requests


In [ ]:
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import requests
from statsmodels.tsa.statespace.sarimax import SARIMAX
from odse.validator import validate_batch

warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/AsobaCloud/odse/main/colab/data"

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#cccccc',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#cccccc',
})

BLUE = '#455BF1'
RED = '#E20419'
DARK = '#1a1a2e'
GREY = '#6b7280'
AMBER = '#f59e0b'
GREEN = '#10b981'


---
## Step 1: Load & Explore the Mzansi Energy Data

The facility's energy management system exports a daily summary with three core metrics:
- `solar_production_kwh` — AC generation from the 2.3 MW rooftop array
- `daily_consumption_kwh` — total site load
- `grid_purchases_kwh` — energy drawn from Eskom when solar falls short

Solar production is highly variable: clear Durban days yield 14,000–18,000 kWh; overcast or fault days can drop below 200 kWh.
The first row (2025-10-30) has a null consumption value — this is the tail end of a hardware fault period documented in the ERP demo.


In [ ]:
energy_balance = pd.read_csv(f"{BASE}/energy_balance.csv")
energy_balance['date'] = pd.to_datetime(energy_balance['date'])

# Drop row(s) with null consumption (hardware fault on 2025-10-30)
energy_balance = energy_balance.dropna(subset=['daily_consumption_kwh']).reset_index(drop=True)

print(f"Rows:       {len(energy_balance):,}")
print(f"Date range: {energy_balance['date'].min().date()} to {energy_balance['date'].max().date()}")
print(f"Solar — mean: {energy_balance['solar_production_kwh'].mean():,.0f} kWh/day  "
      f"max: {energy_balance['solar_production_kwh'].max():,.0f} kWh")
print(f"Consumption mean: {energy_balance['daily_consumption_kwh'].mean():,.0f} kWh/day")
print()
display(energy_balance.head())


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.fill_between(energy_balance['date'], energy_balance['solar_production_kwh'],
                alpha=0.7, color=BLUE, label='Solar Production')
ax.fill_between(energy_balance['date'], energy_balance['grid_purchases_kwh'],
                alpha=0.5, color=GREY, label='Grid Purchases')

ax.set_ylabel('Daily Energy (kWh)')
ax.set_title('Mzansi Energy Balance \u2014 Daily Production vs Grid Purchases\n'
             '2.3 MW Rooftop PV, Durban KZN \u2014 Oct 2025 to May 2026')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()


---
## Step 2: Multi-Year Consumption Trend (Seasonal Context)

Before modelling, it helps to understand the site's seasonal consumption pattern across years.
The monthly aggregate table spans 2023–2026 and reveals a telling anomaly:
**October 2025 consumption was ~839,665 kWh vs 1,222,533 kWh in October 2024** — a 31% drop
consistent with the hardware fault period that reduced measurable load during commissioning of the new monitoring stack.


In [ ]:
consumption_monthly = pd.read_csv(f"{BASE}/mzansi-consumption-monthly-agg.csv")

print("Multi-year monthly consumption (kWh):")
display(consumption_monthly)

print("\nNote: Oct 2025 anomaly (~839,665 kWh vs ~1,222,533 kWh in Oct 2024) reflects the")
print("hardware fault period documented in the Mzansi ERP demo — monitoring data was incomplete.")


---
## Step 3: Normalize to ODS-E Energy Timeseries

Raw site exports use facility-specific column names and no standard for flagging data quality.
ODS-E normalizes heterogeneous site feeds into a common [energy-timeseries](https://opendataschema.energy/docs/schemas/energy-timeseries) schema.

The transform maps:
- `date` → `timestamp` (ISO 8601)
- `solar_production_kwh` → `kWh`
- Adds `direction: "generation"`, `asset_id: "MZN-SITE-001"`
- `error_type`: `"normal"` for days with production > 0, `"offline"` for zero/fault days


In [ ]:
records = [
    {
        'timestamp': row['date'].strftime('%Y-%m-%dT00:00:00+02:00'),
        'kWh': row['solar_production_kwh'],
        'direction': 'generation',
        'asset_id': 'MZN-SITE-001',
        'error_type': 'normal' if row['solar_production_kwh'] > 0 else 'offline',
    }
    for _, row in energy_balance.iterrows()
]

print(f"Total ODS-E records: {len(records):,}")
print(f"Offline days (zero production): {sum(1 for r in records if r['error_type'] == 'offline')}")
print()

import json
print("Sample ODS-E record (clear day):")
clear_day = next(r for r in records if r['kWh'] > 10000)
print(json.dumps(clear_day, indent=2))
print()
print("Sample ODS-E record (fault/offline day):")
offline_day = next(r for r in records if r['error_type'] == 'offline')
print(json.dumps(offline_day, indent=2))


In [ ]:
# Validate the first 100 records against the ODS-E energy-timeseries schema
result = validate_batch(records[:100])
print(result.summary)


---
## Step 4: Fit a Seasonal Forecast Model

Solar production has a clear weekly cycle (weekday vs weekend load patterns affect
inverter clipping behaviour and data logging). We use a
**SARIMAX(1,1,1)(0,1,1,7)** model — a robust seasonal ARIMA with weekly differencing —
to capture both short-run dynamics and the day-of-week seasonal pattern.

Steps:
1. Build a daily `DatetimeIndex` series from the cleaned data (keep first row for any duplicate dates)
2. Reindex to a complete calendar range (forward-fill single missing days)
3. Fit the SARIMAX model
4. Produce a 30-step-ahead forecast with 95% confidence intervals


In [ ]:
# Build daily series — drop duplicate dates, keep first occurrence
ts = (
    energy_balance
    .drop_duplicates(subset='date', keep='first')
    .set_index('date')['solar_production_kwh']
    .sort_index()
    .asfreq('D')
)

# Forward-fill any single-day gaps introduced by asfreq
ts = ts.ffill()

print(f"Training series: {len(ts)} daily observations")
print(f"  from {ts.index.min().date()} to {ts.index.max().date()}")
print(f"  mean: {ts.mean():,.0f} kWh  std: {ts.std():,.0f} kWh")
print()

# Fit SARIMAX(1,1,1)(0,1,1,7)
print("Fitting SARIMAX(1,1,1)(0,1,1,7) — this may take ~30 seconds...")
model = SARIMAX(
    ts,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
fit = model.fit(disp=False)

print(f"AIC: {fit.aic:.1f}   BIC: {fit.bic:.1f}")
print("Model fit complete.")


---
## Step 5: Visualize the 30-Day Forecast

Plot the last 60 days of actuals alongside the 30-day forecast and its 95% confidence interval.
The dashed vertical line marks the train/forecast boundary.


In [ ]:
FORECAST_STEPS = 30
forecast = fit.get_forecast(steps=FORECAST_STEPS)
fc_mean = forecast.predicted_mean
fc_ci = forecast.conf_int(alpha=0.05)

# Clip negative forecast values (solar can't be negative)
fc_mean = fc_mean.clip(lower=0)
fc_ci = fc_ci.clip(lower=0)

# Last 60 days of actuals
actuals_60 = ts.iloc[-60:]
cutoff = ts.index[-1]

fig, ax = plt.subplots(figsize=(14, 6))

# Actuals as bars
ax.bar(actuals_60.index, actuals_60.values, color=BLUE, alpha=0.75,
       width=0.8, label='Actuals (last 60 days)')

# Confidence interval shading
ax.fill_between(
    fc_ci.index,
    fc_ci.iloc[:, 0],
    fc_ci.iloc[:, 1],
    alpha=0.2, color=AMBER, label='95% Confidence Interval'
)

# Forecast line
ax.plot(fc_mean.index, fc_mean.values, color=AMBER, linewidth=2.5,
        marker='o', markersize=4, label='30-Day Forecast')

# Train/forecast boundary
ax.axvline(x=cutoff, color=DARK, linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(cutoff, ax.get_ylim()[1] * 0.95, '  Forecast start',
        fontsize=9, color=DARK, va='top')

# Annotate peak forecast day
peak_date = fc_mean.idxmax()
peak_val = fc_mean.max()
ax.annotate(
    f'Peak forecast\n{peak_date.strftime("%b %d")}: {peak_val:,.0f} kWh',
    xy=(peak_date, peak_val),
    xytext=(peak_date, peak_val + 2500),
    fontsize=9, color=AMBER, fontweight='bold', ha='center',
    arrowprops=dict(arrowstyle='->', color=AMBER, lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=AMBER, alpha=0.9)
)

ax.set_ylabel('Solar Production (kWh/day)')
ax.set_title(
    'Mzansi Solar Production \u2014 30-Day Forecast\n'
    'SARIMAX(1,1,1)(0,1,1,7) \u00b7 Durban, KZN'
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nForecast summary ({fc_mean.index[0].date()} to {fc_mean.index[-1].date()}):")
print(f"  Mean forecast:  {fc_mean.mean():,.0f} kWh/day")
print(f"  Peak forecast:  {peak_val:,.0f} kWh on {peak_date.date()}")
print(f"  Total forecast: {fc_mean.sum():,.0f} kWh over 30 days")


---
## Step 6: Forecast Value \u2014 Grid Purchase Planning

With a solar forecast in hand, the operations team can plan Eskom grid purchases in advance.
**Expected grid purchases** on any day = site load \u2212 forecast solar (capped at zero — solar never exports to grid at this site).

We use the site's mean daily consumption (~35,000 kWh/day) as the load baseline.
Savings vs no-solar = avoided grid purchases compared to running on grid alone.


In [ ]:
MEAN_DAILY_CONSUMPTION = 35_000  # kWh/day — site load baseline

fc_df = pd.DataFrame({
    'date': fc_mean.index,
    'forecast_solar_kwh': fc_mean.values.round(0),
})
fc_df['expected_grid_kwh'] = (MEAN_DAILY_CONSUMPTION - fc_df['forecast_solar_kwh']).clip(lower=0).round(0)
fc_df['savings_vs_no_solar_kwh'] = (MEAN_DAILY_CONSUMPTION - fc_df['expected_grid_kwh']).round(0)
fc_df['date'] = fc_df['date'].dt.strftime('%Y-%m-%d')

print("30-Day Grid Purchase Plan:")
print(fc_df.to_string(index=False))

total_grid = fc_df['expected_grid_kwh'].sum()
total_savings = fc_df['savings_vs_no_solar_kwh'].sum()
print(f"\nTotal expected grid purchases (30 days): {total_grid:,.0f} kWh")
print(f"Total grid savings vs no-solar (30 days):  {total_savings:,.0f} kWh")


In [ ]:
mean_solar = fc_mean.mean()

bar_colors = [
    GREEN if v <= MEAN_DAILY_CONSUMPTION - mean_solar else AMBER
    for v in fc_df['expected_grid_kwh'].values
]

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(
    range(len(fc_df)),
    fc_df['expected_grid_kwh'].values,
    color=bar_colors,
    edgecolor='white',
    linewidth=0.5,
    width=0.7
)

ax.set_xticks(range(len(fc_df)))
ax.set_xticklabels(
    [d[5:] for d in fc_df['date']],  # MM-DD
    fontsize=9, rotation=45, ha='right'
)
ax.set_ylabel('Expected Grid Purchases (kWh)')
ax.set_title(
    'Mzansi \u2014 Expected Grid Purchases Over 30-Day Forecast Window\n'
    'Green = below average grid draw  \u00b7  Amber = above average grid draw'
)

# Reference line: mean grid draw across forecast
mean_grid = fc_df['expected_grid_kwh'].mean()
ax.axhline(y=mean_grid, color=GREY, linestyle='--', alpha=0.6, linewidth=1)
ax.text(len(fc_df) - 0.5, mean_grid + 200,
        f'30-day mean: {mean_grid:,.0f} kWh', fontsize=9, color=GREY, ha='right')

# Legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=GREEN, label='Below average grid draw (good solar day forecast)'),
    Patch(facecolor=AMBER, label='Above average grid draw (low solar day forecast)'),
]
ax.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.show()

green_days = sum(1 for c in bar_colors if c == GREEN)
amber_days = len(bar_colors) - green_days
print(f"\nForecast: {green_days} low-grid days (\u2264 mean draw), {amber_days} high-grid days (> mean draw)")
print(f"Plan bulk grid purchases on the {amber_days} high-draw days to optimise demand charges.")


---
## Summary

This notebook walked through a complete solar forecasting workflow using real Mzansi site data:

1. **Loaded** 6+ months of daily production & consumption data and handled the Oct 2025 hardware fault gracefully
2. **Contextualized** the site against multi-year consumption aggregates, spotting the anomaly that corroborates the ERP fault record
3. **Normalized** all production records to the ODS-E `energy-timeseries` schema — making them joinable with any other ODS-E dataset
4. **Fit** a SARIMAX(1,1,1)(0,1,1,7) model capturing daily solar dynamics and weekly seasonality
5. **Produced** a 30-day forecast with confidence intervals and translated it into an actionable grid purchase plan

**Next steps:**
- [ODS-E Schema Docs](https://opendataschema.energy/docs/schemas/overview) \u2014 Full schema reference
- [Energy Timeseries Schema](https://opendataschema.energy/docs/schemas/energy-timeseries) \u2014 All `energy-timeseries` fields
- [Demand Charge Patterns](https://opendataschema.energy/docs/patterns/demand-charge-optimization) \u2014 Reduce peak demand with forecast-driven dispatch
- [ODS-E on GitHub](https://github.com/AsobaCloud/odse) \u2014 Schema definitions, transforms, and validation tools
